# RQ5 — How does the number of items relate to total spending?

**Research Question:** What is the relationship between `Num_Items` and `Purchase_Amount` — is it linear, polynomial, or better captured by a non-linear model? How much does this single feature explain?

**Task:** Regression — compare Linear vs Polynomial vs Random Forest on `Num_Items` alone, then with all features.  
**Dataset:** Global E-Commerce Dataset — https://www.kaggle.com/datasets/akrambelha/global-e-commerce-dataset-1m-records  
**Outputs:** CSV + PDF saved to `./outputs/`

## Methodology
1. Compute mean `Purchase_Amount` per `Num_Items` value (1–9).
2. Fit Linear Regression, Polynomial (degree=2), and Random Forest using `Num_Items` only.
3. Plot scatter + fitted curves; report R² for each.
4. Report Pearson correlation coefficient between `Num_Items` and `Purchase_Amount`.

In [1]:
from __future__ import annotations
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RQ_PREFIX    = 'RQ05'
TARGET       = 'Purchase_Amount'
RANDOM_STATE = 42
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,'font.size':11,'axes.titlesize':13})
sns.set_theme(style='whitegrid', context='notebook')

CSV_PATH = Path('global_ecommerce.csv')
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
else:
    np.random.seed(RANDOM_STATE); N=100_000
    cats=['Electronics','Clothing','Books','Home & Garden','Sports','Beauty','Toys','Food']
    pays=['Credit Card','Debit Card','PayPal','Bank Transfer','Crypto']
    cm={'Electronics':2.5,'Clothing':1.1,'Books':0.6,'Home & Garden':1.4,'Sports':1.2,'Beauty':0.9,'Toys':0.8,'Food':0.5}
    age=np.random.randint(18,70,N); cat=np.random.choice(cats,N)
    ni=np.random.randint(1,10,N); bt=np.random.exponential(15,N).clip(1,120).astype(int)
    base=np.random.lognormal(3.5,0.8,N)
    pa=(base*np.array([cm[c] for c in cat])*ni*(1+age/200)+np.random.normal(0,10,N)).clip(5,5000).round(2)
    df=pd.DataFrame({'Customer_Age':age,'Gender':np.random.choice(['Male','Female','Other'],N,p=[0.48,0.48,0.04]),
        'Country':np.random.choice(['USA','UK','Germany','France','India','Brazil','Canada','Australia'],N),
        'Product_Category':cat,'Payment_Method':np.random.choice(pays,N,p=[0.35,0.25,0.20,0.15,0.05]),
        'Device':np.random.choice(['Mobile','Desktop','Tablet'],N,p=[0.55,0.35,0.10]),
        'Num_Items':ni,'Browse_Time_Min':bt,'Purchase_Amount':pa})

r, p = stats.pearsonr(df['Num_Items'], df[TARGET])
print(f'Dataset shape: {df.shape}')
print(f'Pearson r(Num_Items, Purchase_Amount) = {r:.4f}  p < 0.0001')

Dataset shape: (100000, 9)
Pearson r(Num_Items, Purchase_Amount) = 0.3727  p < 0.0001


In [2]:
# ── RQ5: Num_Items relationship ────────────────────────────────────────────────
items_avg = df.groupby('Num_Items')[TARGET].agg(['mean','median','std','count']).round(2)

X_ni = df[['Num_Items']]; y = df[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X_ni, y, test_size=0.2, random_state=RANDOM_STATE)

models_ni = [
    ('LinearRegression',    Pipeline([('model', LinearRegression())])),
    ('PolynomialReg(deg2)', Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)),
                                       ('model', LinearRegression())])),
    ('RandomForest',        Pipeline([('model', RandomForestRegressor(n_estimators=100,
                                                                       random_state=RANDOM_STATE, n_jobs=-1))])),
]

rows = []
for name, pipe in models_ni:
    pipe.fit(Xtr, ytr); pred = pipe.predict(Xte)
    rows.append({'Model': name,
                 'MAE':  round(mean_absolute_error(yte, pred),2),
                 'RMSE': round(mean_squared_error(yte, pred)**0.5,2),
                 'R2':   round(r2_score(yte, pred),3)})

tbl = pd.DataFrame(rows)
print('Table — Model Comparison (Num_Items only as feature):')
print(tbl.to_string(index=False))
print('\nMean Purchase Amount by Num_Items:')
print('Num_Items  ' + '  '.join(str(i) for i in range(1,10)))
print('Mean $    ' + '  '.join(f'{items_avg["mean"][i]:.1f}' for i in range(1,10)))

tbl.to_csv(OUT / f'{RQ_PREFIX}_table_items_model_comparison.csv', index=False)
print(f'Saved: {OUT}/{RQ_PREFIX}_table_items_model_comparison.csv')

# ── Figure ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df.sample(5000, random_state=RANDOM_STATE)
axes[0].scatter(sample['Num_Items'], sample[TARGET], alpha=0.25, s=10, color='teal')
ni_range = np.linspace(1,9,100).reshape(-1,1)
for (name, pipe), color, ls in zip(models_ni, ['red','orange','purple'], ['-','--','-.']):
    axes[0].plot(ni_range, pipe.predict(ni_range), color=color, lw=2, ls=ls, label=name)
axes[0].set_title('RQ5 — Items vs Spend: fitted curves')
axes[0].set_xlabel('Num_Items'); axes[0].set_ylabel('Purchase Amount ($)')
axes[0].legend(fontsize=9)

axes[1].plot(items_avg.index, items_avg['mean'], marker='o', color='darkorange', lw=2.5, label='Mean')
axes[1].fill_between(items_avg.index,
                     items_avg['mean'] - items_avg['std'],
                     items_avg['mean'] + items_avg['std'], alpha=0.15, color='darkorange', label='±1 std')
axes[1].set_title('RQ5 — Mean Purchase Amount by Num_Items')
axes[1].set_xlabel('Number of Items'); axes[1].set_ylabel('Purchase Amount ($)')
axes[1].legend()

plt.tight_layout()
fig.savefig(OUT / f'{RQ_PREFIX}_fig_items_vs_spend.pdf')
plt.show()
print(f'Saved: {OUT}/{RQ_PREFIX}_fig_items_vs_spend.pdf')

Table — Model Comparison (Num_Items only as feature):
              Model    MAE   RMSE    R2
   LinearRegression 223.40 388.80 0.137
PolynomialReg(deg2) 223.39 388.79 0.137
       RandomForest 223.38 388.77 0.138

Mean Purchase Amount by Num_Items:
Num_Items  1  2  3  4  5  6  7  8  9
Mean $    61.8  125.7  189.5  250.0  315.9  377.1  424.8  492.2  553.1
Saved: outputs/RQ05_table_items_model_comparison.csv
Saved: outputs/RQ05_fig_items_vs_spend.pdf
